# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", getattr(metadata, "name", "N/A"))
print("Dataset Description: ", getattr(metadata, "description", "N/A"))

## 2. Data Overview
Review available record sets, fields, and their IDs (all referenced by their `@id`).

In [ ]:
# List record sets and their field @ids

record_sets = list(dataset.record_sets())
print(f"Total record sets: {len(record_sets)}\n")

record_set_ids = []
for i, rec in enumerate(record_sets):
    print(f"Record Set {i+1} @id: {rec['@id']}")
    record_set_ids.append(rec["@id"])
    # Show fields for this record set
    print("  Fields (@id):")
    for field in rec.get("fields", []):
        print(f"   - {field['@id']} (name: {field.get('name','')})")
    print()
# If there's only one record set, pick it for downstream usage
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (referenced by @id)
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set @id '{record_set_id}'")
    except Exception as e:
        print(f"Error loading records for record set {record_set_id}: {e}")

if selected_record_set_id is not None and selected_record_set_id in dataframes:
    print("\nColumns in DataFrame for record set @id:", selected_record_set_id)
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

> **Note**: All fields referenced by their `@id`. Adjust your numeric_field_id and group_field_id accordingly from the overview.

In [ ]:
# Set example IDs for numeric field and group field based on record set overview (edit as needed)

# Example placeholder, update with the real field @id you want to explore
example_numeric_field_id = None
example_group_field_id = None

if selected_record_set_id is not None and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Try to automatically detect a numeric field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric_field_id = col
            break
    # Try a 'group' field (e.g. sample, type, category)
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and 'sample' in col.lower():
            example_group_field_id = col
            break

if example_numeric_field_id is None:
    print("No numeric field found. Please set 'example_numeric_field_id' to a column name representing a numeric field.")
else:
    print(f"Using numeric field: {example_numeric_field_id}")

if example_numeric_field_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Drop NaNs for filtering
    numeric_series = pd.to_numeric(df[example_numeric_field_id], errors='coerce').fillna(0)
    df[example_numeric_field_id] = numeric_series
    threshold = numeric_series.mean()  # As an example, use mean as filter threshold
    filtered_df = df[df[example_numeric_field_id] > threshold].copy()
    print(f"Filtered records with {example_numeric_field_id} > mean ({threshold:.2f}):")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{example_numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"Normalized '{example_numeric_field_id}' for filtered records:")
    display(filtered_df[[example_numeric_field_id, norm_col]].head())

    if example_group_field_id and example_group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{example_group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and selected_record_set_id in dataframes and example_numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_record_set_id][example_numeric_field_id], bins=50, kde=True)
    plt.title(f"Distribution for field '@id': {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.show()

    if example_group_field_id and example_group_field_id in dataframes[selected_record_set_id].columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=dataframes[selected_record_set_id][example_numeric_field_id], x=dataframes[selected_record_set_id][example_group_field_id])
        plt.title(f"Boxplot of {example_numeric_field_id} by {example_group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides protein abundances and properties for extracellular vesicles from stimulated human mast cells, as revealed by mass spectrometry.
- After loading and filtering the data, we visualized and grouped key numeric fields (e.g., abundance, molecular weight) by their categorical attributes such as sample or type, if present.
- This approach, referencing all schema elements by their `@id` (as required in Croissant), ensures reproducibility and precise mapping between metadata and analysis steps.